MuJoCo Locomotion Task

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
import h5py
import numpy as np
import gymnasium as gym

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device =", dev)
if dev == "cuda":
    print("gpu  =", torch.cuda.get_device_name(0))

In [ ]:
# references from https://github.com/tinkoff-ai/d4rl/blob/master/d4rl/infos.py
# used to normalize scores later
REF_MIN = -280.178953
REF_MAX = 12135.0
# from decision transformer paper
TARGET = 6000.0
SCALE = 1000.0

env = gym.make("HalfCheetah-v4")

# read in file
PATH = "halfcheetah_medium-v2.hdf5"
with h5py.File(PATH, "r") as f:
    S = f["observations"][:] # states
    A = f["actions"][:]
    R = f["rewards"][:]
    end_flag = np.logical_or(f["terminals"][:], f["timeouts"][:]) # end_flag either due to robot falling over or max step limit

# format data into trajectories
trajectories = []
i = 0
for j in range(len(end_flag)):
    if end_flag[j]:
        trajectories.append({"S": S[i:j + 1], "A": A[i:j + 1], "R": R[i:j + 1]})
        i = j + 1

S_dim = trajectories[0]["S"].shape[1]
A_dim = trajectories[0]["A"].shape[1]
states = np.concatenate([t["S"] for t in trajectories])

# normalization
s_mean = states.mean(axis = 0)
s_std = states.std(axis = 0) + 1e-7
for t in trajectories:
    t["S"] = ((t["S"] - s_mean) / s_std).astype(np.float32)
    t["A"] = t["A"].astype(np.float32)
    t["RTG"] = (np.cumsum(t["R"][::-1])[::-1] / SCALE).astype(np.float32) # from decision transformer paper

# essentially softmax based on trajectory lengths, used when generating batches
lengths = np.array([len(t["R"]) for t in trajectories])
probs = lengths / lengths.sum()

In [ ]:
# batch size, K specified in decision transformer
# this function was modified directly from decision transformer
# randomly select trajectory, slice random window of size K and learn by predicting next action
def get_batch(batch_size = 64, K = 20):
    batch_inds = np.random.choice(len(trajectories), size = batch_size, p = probs)
    s, a, rtg, timesteps, mask = [], [], [], [], []
    for i in range(batch_size):
        traj = trajectories[batch_inds[i]]
        si = np.random.randint(0, len(traj["R"]))

        s.append(traj["S"][si:si + K].reshape(1, -1, S_dim))
        a.append(traj["A"][si:si + K].reshape(1, -1, A_dim))
        rtg.append(traj["RTG"][si:si + K].reshape(1, -1, 1))
        timesteps.append(np.arange(si, si + s[-1].shape[1]).reshape(1, -1))

        tlen = s[-1].shape[1]
        s[-1] = np.concatenate([np.zeros((1, K - tlen, S_dim)), s[-1]], axis = 1)
        a[-1] = np.concatenate([np.ones((1, K - tlen, A_dim)) * -10.0, a[-1]], axis = 1)
        rtg[-1] = np.concatenate([np.zeros((1, K - tlen, 1)), rtg[-1]], axis = 1)
        timesteps[-1] = np.concatenate([np.zeros((1, K - tlen)), timesteps[-1]], axis = 1)
        mask.append(np.concatenate([np.zeros((1, K - tlen)), np.ones((1, tlen))], axis = 1))

    s = torch.from_numpy(np.concatenate(s, axis = 0)).to(dtype = torch.float32, device = dev)
    a = torch.from_numpy(np.concatenate(a, axis = 0)).to(dtype = torch.float32, device = dev)
    rtg = torch.from_numpy(np.concatenate(rtg, axis = 0)).to(dtype = torch.float32, device = dev)
    timesteps = torch.from_numpy(np.concatenate(timesteps, axis = 0)).to(dtype = torch.long, device = dev)
    mask = torch.from_numpy(np.concatenate(mask, axis = 0)).to(dtype = torch.float32, device = dev)

    return rtg, s, a, timesteps, mask

In [ ]:
# RNN -> MLP, this will replace self-attention in decision transformer
class minRNN(nn.Module):
    def __init__(self, dim, cell, dropout = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.rnn = cell(dim, hidden_dim = dim * 4, proj_out = True)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.RELU(), nn.Linear(dim * 4, dim))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.rnn(self.ln1(x)))
        x = x + self.drop(self.mlp(self.ln2(x)))
        return x

# this code was modified directly from decision transformer
class DecisionRNN(nn.Module):
    def __init__(self, state_dim, act_dim, hidden_size, cell, n_layers = 3, dropout = 0.1, max_ep_len  = 1001, action_tanh = True):
        super().__init__()
        self.state_dim = state_dim
        self.act_dim = act_dim
        self.hidden_size = hidden_size
        self.embed_timestep = nn.Embedding(max_ep_len, hidden_size)
        self.embed_return = nn.Linear(1, hidden_size)
        self.embed_state = nn.Linear(state_dim, hidden_size)
        self.embed_action = nn.Linear(act_dim, hidden_size)
        self.embed_ln = nn.LayerNorm(hidden_size)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(hidden_size, cell, dropout = dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(hidden_size)
        self.predict_action = nn.Sequential(
            *([nn.Linear(hidden_size, act_dim)] + ([nn.Tanh()] if action_tanh else []))
        )

    def forward(self, R, S, A, T):
        batch_size, seq_length = states.shape[0], states.shape[1]

        time_embeddings = self.embed_timestep(timesteps)
        state_embeddings = self.embed_state(states) + time_embeddings
        action_embeddings = self.embed_action(actions) + time_embeddings
        returns_embeddings = self.embed_return(returns_to_go) + time_embeddings

        x = torch.stack(
            (returns_embeddings, state_embeddings, action_embeddings), dim = 1).permute(0, 2, 1, 3).reshape(batch_size, 3 * seq_length, self.hidden_size)
        x = self.drop(self.embed_ln(x))

        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)

        x = x.reshape(batch_size, seq_length, 3, self.hidden_size).permute(0, 2, 1, 3)
        return self.predict_action(x[:, 1])

In [ ]:
@torch.no_grad()
def evaluate(model, env, n_episodes = 5, target = TARGET, K = 20):
    model.eval()
    returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed = ep)
        states, acts, rtgs, steps = [], [], [], []
        rtg, ret = target / SCALE, 0.0
        for t in range(1000):
            states.append(((s - s_mean) / s_std).astype(np.float32))
            acts.append(np.zeros(A_dim, dtype = np.float32))
            rtgs.append(rtg)
            steps.append(t)
            S = torch.tensor(np.array(states[-K:]), device = dev)[None]
            A = torch.tensor(np.array(acts[-K:]), device = dev)[None]
            R = torch.tensor(rtgs[-K:], dtype = torch.float32, device = dev)[None, :, None]
            T = torch.tensor(steps[-K:], dtype = torch.long, device = dev)[None]
            a = model(R, S, A, T)[0, -1].cpu().numpy()
            acts[-1] = a
            s, r, terminated, truncated, _ = env.step(a)
            ret += r
            rtg -= r / SCALE
            if terminated or truncated:
                break
        returns.append(100 * (ret - REF_MIN) / (REF_MAX - REF_MIN))
    model.train()
    return float(np.mean(returns))

model = DecisionRNN(S_dim, A_dim, dim = 128, cell = minLSTM).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr = 1e-3, weight_decay = 1e-4)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda step: min((step + 1) / 10000, 1.0))

for step in range(100000):
    R, S, A, T, M = get_batch()
    pred = model(R, S, A, T)
    loss = (((pred - A) ** 2).mean(dim = -1) * M).sum() / M.sum()
    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 0.25)
    opt.step()
    sched.step()
    if (step + 1) % 10000 == 0:
        print(step + 1, round(loss.item(), 4), round(evaluate(model, env), 1))